# 03 - Model training

**Goal.** Run the three feature experiments across four models for each
component, without leaking test information, and select a final model by a rule
fixed in advance.

The full grid is 4 components x 3 feature sets x 4 models = **48
cross-validated experiments**. `src/train.py` runs all of them; this notebook
explains and reproduces the mechanics on one component.

In [ ]:
import sys
from pathlib import Path

# The notebooks live in notebooks/; the modules live in src/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)
print("project root:", PROJECT_ROOT)

## 3.1 The leakage problem, stated precisely

Every step that *learns something from data* — the median used for imputation,
the mean and scale used for standardisation, the correlation threshold, the
mutual-information ranking, the tree importances — must be estimated from
training data only. Computing any of them on the full table before splitting
means information from the test cycles has already reached the model, and the
reported score is optimistic.

The defence here is **structural, not procedural**: all learned steps are stages
of one `sklearn.pipeline.Pipeline`, and only that object is passed to
`cross_validate` / `fit`. scikit-learn then re-fits every stage on each training
fold automatically. Leaking by accident becomes impossible.

In [ ]:
from train import build_experiment_pipeline

pipeline = build_experiment_pipeline("selected", "random_forest", class_weight="balanced")
for name, step in pipeline.steps:
    print(f"  {name:<22}{type(step).__name__}")

Note the ordering. The correlation filter runs **before** the scaler because
Pearson *r* is scale-invariant — filtering first saves the scaler from fitting
hundreds of columns that are about to be discarded. The supervised stages
(mutual information, then tree importance) run after scaling and get
progressively more expensive and more multivariate:

1. **Correlation filter** — unsupervised, removes redundancy.
2. **Mutual information** — supervised, univariate, captures *non-linear*
   dependence that a correlation-with-label would score at zero.
3. **Tree importance** — supervised, *multivariate*: scores each feature in the
   presence of the others, so a feature useful only in combination survives.

## 3.2 Loading the feature tables

In [ ]:
from data_loader import TARGET_SPECS, load_profile
from preprocessing import feature_columns, describe_class_balance, resolve_class_weight
from utils import FEATURES_DIR, PROCESSED_DATA_DIR, RANDOM_STATE

# Produced by src/train.py. Run it first if these files are missing.
engineered = pd.read_csv(FEATURES_DIR / "engineered_features.csv")
baseline = pd.read_csv(PROCESSED_DATA_DIR / "baseline_features.csv")
profile = load_profile()

print("engineered:", engineered.shape, " baseline:", baseline.shape)

**The baseline representation** is the per-sensor cycle mean — 17 features. It is
the minimum transformation that makes the data rectangular at all: the three
sampling rates mean *some* aggregation is unavoidable, and taking the mean is the
most obvious choice an engineer would make without doing any feature
engineering. Comparing against it isolates the contribution of the engineered
blocks rather than the contribution of "having a table".

## 3.3 One component end to end: the valve

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import LabelEncoder

COMPONENT = "valve"
spec = TARGET_SPECS[COMPONENT]

y_raw = profile[spec.column].to_numpy()
encoder = LabelEncoder().fit(y_raw)
y = encoder.transform(y_raw)

balance = describe_class_balance(y_raw, spec.value_labels)
class_weight = resolve_class_weight(y)
print(balance.to_string(index=False))
print("\nimbalance ratio:", round(balance.attrs['imbalance_ratio'], 2))
print("class weighting :", class_weight)

**Why class weighting rather than SMOTE.** These classes are *ordered
degradation levels* of one physical component. SMOTE interpolates between
neighbouring feature vectors, so it would manufacture cycles halfway between
"small lag" and "severe lag" — cycles the rig never produced, sitting exactly
where the decision boundary is hardest. Class weighting rebalances the objective
without inventing data.

In [ ]:
# 80/20 stratified split. The test indices are set aside and not touched again
# in this notebook.
train_idx, test_idx = train_test_split(
    np.arange(len(y)), test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"train {len(train_idx)} cycles | test {len(test_idx)} cycles (held out)")

In [ ]:
from train import MODEL_NAMES, FEATURE_SETS

tables = {"baseline": baseline, "engineered": engineered, "selected": engineered}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []

for feature_set in FEATURE_SETS:
    table = tables[feature_set]
    columns = feature_columns(table)
    X = table[columns].to_numpy(float)
    for model_name in MODEL_NAMES:
        pipe = build_experiment_pipeline(feature_set, model_name, class_weight)
        scores = cross_validate(pipe, X[train_idx], y[train_idx], cv=cv,
                                scoring=["accuracy", "f1_macro"], n_jobs=1)
        rows.append({
            "feature_set": feature_set, "model": model_name,
            "accuracy": scores["test_accuracy"].mean(),
            "f1_macro": scores["test_f1_macro"].mean(),
            "f1_macro_std": scores["test_f1_macro"].std(),
            "fit_s": scores["fit_time"].mean(),
        })
        print(f"{feature_set:<12}{model_name:<22}macro F1 = {rows[-1]['f1_macro']:.4f}")

results = pd.DataFrame(rows)

In [ ]:
pivot = results.pivot(index="model", columns="feature_set", values="f1_macro")
pivot = pivot[[c for c in FEATURE_SETS if c in pivot.columns]]
display(pivot.round(4))

ax = pivot.plot.bar(figsize=(9, 4.5), rot=0)
ax.set_ylabel("cross-validated macro F1")
ax.set_ylim(0, 1.05)
ax.set_title(f"{spec.display_name}: effect of feature engineering and selection")
ax.legend(title="feature set")

**The result worth explaining in a viva.** Feature *selection* helps the weaker
learners most. SVM and Logistic Regression are distance- and coefficient-based,
so hundreds of redundant, collinear features actively hurt them; the trees are
far more robust to that. Removing redundancy is what makes the margin-based
model competitive — which is a much more interesting finding than "the boosted
trees won".

## 3.4 The selection rule

In [ ]:
best = results.loc[results["f1_macro"].idxmax()]
print("Selection rule: highest mean cross-validated macro F1 on the TRAINING set.")
print(f"Selected: {best['model']} on '{best['feature_set']}' "
      f"(macro F1 = {best['f1_macro']:.4f} +/- {best['f1_macro_std']:.4f})")

The rule is fixed **before** looking at any test score, and applied identically
to all four components. Macro F1 rather than accuracy, because it weights every
class equally: a model cannot look good by ignoring a rare but operationally
critical failure state.

The selected configuration is then refit on the whole training set and evaluated
**once** on the untouched test set — in `src/evaluate.py`, deliberately a
separate script so that no tuning loop can reach the test data.

## 3.5 Tracing which features actually reach the classifier

In [ ]:
from feature_selection import trace_selected_feature_names, selection_stage_counts

table = tables[best["feature_set"]]
columns = feature_columns(table)
X = table[columns].to_numpy(float)

final = build_experiment_pipeline(best["feature_set"], best["model"], class_weight)
final.fit(X[train_idx], y[train_idx])

print(selection_stage_counts(final, len(columns)))
selected = trace_selected_feature_names(final, columns)
print(f"\n{len(selected)} features reach the classifier. First 15:")
for name in selected[:15]:
    print("  ", name)

Each selector returns a bare numpy array, so after three chained stages the model
sees anonymous column indices. `trace_selected_feature_names` composes the masks
in order to recover the names — which is what makes the dashboard's
explainability panel and the importance figures possible at all.